# Fine-tuning tactile paving v3 dengan data publik

Notebook ini mengunduh subset GuideTWSI RBar yang dibatasi, membangun campuran 4:1 dan 2:1, lalu melatih dua kandidat dari checkpoint GuideTWSI. Protected ground truth tidak disentuh di notebook ini. Output tetap kandidat offline, bukan izin penggunaan keselamatan.

In [ ]:
!pip install -q "ultralytics==8.4.138" "kagglehub==1.0.2" opencv-python-headless
!command -v git-lfs >/dev/null || (apt-get update -qq && apt-get install -y -qq git-lfs)
!git lfs install

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO = Path('/content/Yoloooo')
BRANCH = 'codex/model-first-mobile-ready'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', 'https://github.com/Riqqi15/Yoloooo.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH], check=True)
subprocess.run(['git', '-C', str(REPO), 'lfs', 'pull'], check=True)

In [ ]:
def run_script(name, *arguments):
    subprocess.run([sys.executable, str(REPO / 'scripts' / name), *map(str, arguments)], cwd=REPO, check=True)

inventory = REPO / 'runs/public-data-cache/guidetwsi-inventory.json'
provenance = REPO / 'data/public/guidetwsi-rbar-v1/provenance.json'
cache = REPO / 'runs/public-data-cache/guidetwsi-rbar-v1'
if not inventory.exists():
    run_script('acquire_guidetwsi_subset.py', 'inventory', '--output', inventory)
if not provenance.exists() or not cache.exists() or not any(cache.rglob('*')):
    run_script('acquire_guidetwsi_subset.py', 'download', '--inventory', inventory, '--max-bytes', '5000000000', '--max-images', '2000', '--workers', '8')
run_script('prepare_guidetwsi_subset.py', '--provenance', provenance, '--cache', cache, '--protected-dir', REPO / 'data/samples', '--existing-images', REPO / 'artifacts/datasets/station-tactile-v2/tactile/images', '--limit', '2000', '--output', REPO / 'data/training/manifests/guidetwsi-rbar-2k-v1.json')

In [ ]:
for ratio in (4, 2):
    version = f'tactile-v3-public{ratio}-station1'
    output = REPO / 'artifacts/datasets' / version
    manifest = REPO / 'data/training/manifests' / f'{version}.json'
    if not output.exists():
        run_script('build_tactile_v3_dataset.py', '--output-root', output, '--manifest-output', manifest, '--public-to-station', str(ratio))

In [ ]:
for ratio in (4, 2):
    version = f'tactile-v3-public{ratio}-station1'
    run_name = f'tactile-one-class-v3-public{ratio}-station1'
    candidate = REPO / 'artifacts/candidates' / run_name
    if not candidate.exists():
        run_script('train_tactile_v3.py', '--dataset', REPO / 'artifacts/datasets' / version / 'dataset.yaml', '--checkpoint', REPO / 'models/guidetwsi/yolo11n_tactile.pt', '--run-name', run_name, '--runs-root', REPO / 'runs/segment', '--candidate-root', REPO / 'artifacts/candidates', '--device', '0', '--epochs', '80', '--batch', '8')

In [ ]:
import shutil
from google.colab import files

for ratio in (4, 2):
    run_name = f'tactile-one-class-v3-public{ratio}-station1'
    archive = shutil.make_archive(f'/content/{run_name}', 'zip', REPO / 'artifacts/candidates' / run_name)
    files.download(archive)